# 🔥 Advanced · DPO — preference alignment

> 🔥 **Advanced / heavy lab.** Teach an LLM which answers people prefer, using chosen/rejected pairs (DPO — the RLHF-free recipe).
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **20–40 min** including downloads. Built on the official **[TRL DPOTrainer](https://huggingface.co/docs/trl/dpo_trainer)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Run this *after* SFT (the QLoRA lab) — alignment is the second stage of the modern LLM pipeline.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — small model | A100 40/80 GB (policy + reference) |
| **Storage** | model + preference subset | UltraFeedback ~ GBs; 50–100 GB disk |
| **Time** | 200 steps ~ 15–30 min | full set ~ several hours (≈1.5–2× SFT) |

**Full pipeline (scale-up):** `accelerate launch dpo.py …` on full preference data after SFT.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

## 1 · Load the (SFT) model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
base = "Qwen/Qwen2.5-0.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map="auto")

## 2 · Preference data — prompt + chosen + rejected

In [ ]:
from datasets import load_dataset
ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:2000]")
print(ds.column_names)

## 3 · Train with DPO (LoRA adapters, no separate reward model)

In [ ]:
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig
args = DPOConfig(output_dir="dpo-out", per_device_train_batch_size=2, gradient_accumulation_steps=4,
                 max_steps=200, learning_rate=5e-6, logging_steps=20, bf16=True, beta=0.1, report_to="none")
trainer = DPOTrainer(model=model, args=args, train_dataset=ds, processing_class=tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM", target_modules="all-linear"))
trainer.train()

## 4 · Inspect
Watch `rewards/chosen` rise above `rewards/rejected` and the `rewards/accuracies` climb in the logs — that's the model learning the preference. Save:

In [ ]:
trainer.save_model("dpo-adapter")

## Evaluate — preference accuracy on a held-out split
How often the aligned model scores the *chosen* answer above the *rejected* one (1.0 = always).

In [ ]:
from datasets import load_dataset
ev = load_dataset("trl-lib/ultrafeedback_binarized", split="train[2000:2300]")
m = trainer.evaluate(eval_dataset=ev)
print({k: round(v, 4) for k, v in m.items() if "reward" in k or "accurac" in k})   # eval_rewards/accuracies

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
Preference alignment generalises beyond text:
- **A · Human** align a motion generator (MDM) to preferred motions · **D · Scene / world** preference / RL-style alignment for agents and world-model planning.

## Notes & next steps
- DPO needs an SFT'd base; using a raw base model gives weak results.
- Variants: **IPO**, **KTO**, **ORPO** (one-stage) — all in TRL.
- Lower `beta` = trust the data more; raise it = stay closer to the reference model.